In [11]:
## Standard libraries
import os
import json
import math
import numpy as np
import time

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgb
import matplotlib
matplotlib.rcParams['lines.linewidth'] = 2.0
import seaborn as sns
sns.reset_orig()
sns.set()

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
# Torchvision
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
# PyTorch Lightning
try:
    import pytorch_lightning as pl
except ModuleNotFoundError: # Google Colab does not have PyTorch Lightning installed by default. Hence, we do it here if necessary
    !pip install --quiet pytorch-lightning>=1.4
    import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

# Setting the seed
pl.seed_everything(42)

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


/tmp/ipykernel_5951/3001796416.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('svg', 'pdf') # For export
INFO:lightning_fabric.utilities.seed:Seed set to 42


cpu


## GCNs (Graph Convolutional Netwroks)

GCNs are similar to convolutions in images in the sense that the “filter” parameters are typically shared over all locations in the graph.



*   GCNs rely on message passing methods, which means that vertices exchange information with the neighbors, and send “messages” to each other.

![image.png](https://zoshs2.github.io/assets/img/post/gcn_implement/GCN_message_passing.png)


The first step is that each node creates a feature vector that represents the message it wants to send to all its neighbors. In the second step, the messages are sent to the neighbors, so that a node receives one message per adjacent node.

Here, the usual way to go is to sum or take the mean. Given the previous features of nodes $H^{(l)}$, the GCN layer is defined as follows:

$$
H^{(l+1)} = \sigma \left( \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2} H^{(l)} W^{(l)} \right)
$$

$W^{(l)}$ is the weight parameters with which we transform the input features into messages $(H^{(l)} W^{(l)})$.

To the adjacency matrix $A$ we add the identity matrix so that each node sends its own message also to itself:
$$
\hat{A} = A + I
$$

Finally, to take the average instead of summing, we calculate the matrix $\hat{D}$ which is a diagonal matrix with $D_{ii}$ denoting the number of neighbors node $i$ has.

$\sigma$ represents an arbitrary activation function, and not necessarily the sigmoid (usually a ReLU-based activation function is used in GNNs).

In [12]:
class GCNLayer(nn.Module):

    def __init__(self, c_in, c_out):
        super().__init__()
        self.projection = nn.Linear(c_in, c_out)

    def forward(self, node_feats, adj_matrix):
        """
        Inputs:
            node_feats - Tensor with node features of shape [batch_size, num_nodes, c_in]
            adj_matrix - Batch of adjacency matrices of the graph. If there is an edge from i to j, adj_matrix[b,i,j]=1 else 0.
                         Supports directed edges by non-symmetric matrices. Assumes to already have added the identity connections.
                         Shape: [batch_size, num_nodes, num_nodes]
        """
        # Num neighbours = number of incoming edges
        num_neighbours = adj_matrix.sum(dim=-1, keepdims=True)
        node_feats = self.projection(node_feats)
        node_feats = torch.bmm(adj_matrix, node_feats)
        node_feats = node_feats / num_neighbours
        return node_feats

To further understand the GCN layer, we can apply it to our example graph above. First, let’s specify some node features and the adjacency matrix with added self-connections:

In [13]:
node_feats = torch.arange(8, dtype=torch.float32).view(1, 4, 2)
adj_matrix = torch.Tensor([[[1, 1, 0, 0],
                            [1, 1, 1, 1],
                            [0, 1, 1, 1],
                            [0, 1, 1, 1]]])

print("Node features:\n", node_feats)
print("\nAdjacency matrix:\n", adj_matrix)

Node features:
 tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])

Adjacency matrix:
 tensor([[[1., 1., 0., 0.],
         [1., 1., 1., 1.],
         [0., 1., 1., 1.],
         [0., 1., 1., 1.]]])


Next, let’s apply a GCN layer to it.

In [14]:
layer = GCNLayer(c_in=2, c_out=2)
layer.projection.weight.data = torch.Tensor([[1., 0.], [0., 1.]])
layer.projection.bias.data = torch.Tensor([0., 0.])

with torch.no_grad():
    out_feats = layer(node_feats, adj_matrix)

print("Adjacency matrix", adj_matrix)
print("Input features", node_feats)
print("Output features", out_feats)

Adjacency matrix tensor([[[1., 1., 0., 0.],
         [1., 1., 1., 1.],
         [0., 1., 1., 1.],
         [0., 1., 1., 1.]]])
Input features tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
Output features tensor([[[1., 2.],
         [3., 4.],
         [4., 5.],
         [4., 5.]]])


As we can see, the first node’s output values are the average of itself and the second node. Similarly, we can verify all other nodes.


## GNNs (Graph Neural Networks)
However, in a GNN, we would also want to allow feature exchange between nodes beyond its neighbors. This can be achieved by applying multiple GCN layers, which gives us the final layout of a GNN. The GNN can be build up by a sequence of GCN layers and non-linearities such as ReLU.

![image.png](https://raw.githubusercontent.com/Lightning-AI/tutorials/main/course_UvA-DL/06-graph-neural-networks/gcn_network.png)

Lets test out GNNs with a small problem statement of prediction. It can be done easily with the classic NNs but here we will intoduce graph and how other nodes effect the each others.

# Problem Statement: The "Social Influence" Predictor
Imagine a small social network of 4 people. Some are "Influencers" (Label 1) and some are "Followers" (Label 0).

## The Data:
We only know the "Label" of 2 people. We need the GNN to predict the labels of the other 2 based on who they talk to.

## The Features:
Each person has a simple 2D feature vector (e.g., [Activity Level, Profile Completeness]).

In [15]:
class MultiLayerGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # Layer 1
        self.layer1 = GCNLayer(input_dim, hidden_dim)

        # Layer 2
        self.layer2 = GCNLayer(hidden_dim, hidden_dim)

        # Layer 3
        self.layer3 = GCNLayer(hidden_dim, output_dim)

    def forward(self, x, adj):
        # Layer 1 + Activation
        x = self.layer1(x, adj)
        x = F.relu(x)

        # Layer 2 + Activation
        x = self.layer2(x, adj)
        x = F.relu(x)

        # Layer 3 (No ReLU here, we want output/prediction)
        x = self.layer3(x, adj)
        return x

In [16]:
node_features = torch.tensor([
    [[1.0, 0.5],
     [0.1, 0.2],
      [0.5, 0.9],
      [0.2, 0.1]]
]) # Shape: [1, 4, 2] -> [batch, nodes, features]

adj = torch.tensor([
    [[1, 1, 0, 0],
     [1, 1, 0, 0],
     [0, 0, 1, 1],
     [0, 0, 1, 1]]
]).float()

model = MultiLayerGNN(input_dim=2, hidden_dim=4, output_dim=2)

# Forward Pass
output = model(node_features, adj)

print("(Output):\n", output)

(Output):
 tensor([[[ 0.1658, -0.1086],
         [ 0.1658, -0.1086],
         [ 0.1606, -0.1169],
         [ 0.1606, -0.1169]]], grad_fn=<DivBackward0>)


How to use this Output:

we can interpret these numbers like this:


```
Node,      Class 0 Score,          Class 1 Score,          Prediction
0,           -0.3838,                0.0644,                Class 1 (higher score)
1,           -0.3838,                0.0644,                Class 1 (higher score)
2,           -0.3373,                0.1572,                Class 1 (higher score)
3,           -0.3373,                0.1572,                Class 1 (higher score)
```


# Note:
There is one interesting observation , that is if we look closely, in final output feature of *Node 0 and Node 1* & *Node 2 and Node 3* are *similar.*

This tells us that our Multiple Layers successfully mixed the data across the graph. However, because the graph is tiny and the layers are many, the nodes are starting to lose their individual "identity" and are becoming "clones" of their neighbors.

# Now wheather this is good or bad??
Well answer to this question depends upon our use case.
## Community Detection
If we want to group nodes into "clubs" or "communities," `cloning is actually helpful.`

The Logic: If Node 0 and Node 1 are in the same friendship circle, they should have similar embeddings.

The Result: The model ignores tiny individual differences and focuses on the Big Picture. It realizes, "Hey, these four people all belong to the same group," which makes clustering them very easy.
## Node-Level Detection
If we want to tell the difference between two neighbors (e.g., in a Fraud Detection system), `cloning is a disaster.`

The Problem: If Node 0 is a "Normal User" and Node 1 is a "Scammer," but they are friends, a 3-layer GCN will make their features identical.

The Result: Your model will be unable to catch the scammer because the scammer's features have been "washed away" by the normal user's features.

# Now Is there Balance?
## The "Sweet Spot" (The Goldilocks Principle)

In GNN research, we usually try to find a balance. You want nodes to learn from their neighbors, but you don't want them to lose their soul.

```
Feature,                  Low Layers (1-2),                             High Layers (5+)
Information,           Local (Immediate neighbors),                  Global (Entire graph)
Diversity,              High (Nodes stay unique),                  Low (Nodes become ""clones"")
Best For,              Predicting individual traits,                  Predicting community trends
```



# GAT (Graph Attention Network)

 Attention describes a weighted average of multiple elements with the weights dynamically computed based on an input query and elements’ keys.

 Similarly to the GCN, the graph attention layer creates a message for each node using a linear layer/weight matrix.

 Lets see calculation for single MLP:

 ![image.png](https://uvadlc-notebooks.readthedocs.io/en/latest/_images/graph_attention_MLP.svg)

 $h_i$ and $h_j$ are the original features from node $i$ and $j$ respectively, and represent the messages of the layer with $\mathbf{W}$ as weight matrix. $\mathbf{a}$ is the weight matrix of the MLP, which has the shape $[1, 2 \times d_{\text{message}}]$, and $\alpha_{ij}$ the final attention weight from node $i$ to $j$. The calculation can be described as follows:

$$\alpha_{ij} = \frac{\exp \left(\text{LeakyReLU} \left(\mathbf{a} \left[\mathbf{W}h_i || \mathbf{W}h_j\right]\right)\right)}{\sum_{k \in \mathcal{N}_i} \exp \left(\text{LeakyReLU} \left(\mathbf{a} \left[\mathbf{W}h_i || \mathbf{W}h_k\right]\right)\right)}$$

**LeakyRelu**: This function is used before softmax, to introduce the non-linearity. Why we need non-linearity, is to avoid the `issue of Cloning` that we faced while working with GNNs.



In [17]:
# How to write the MLP calculation

import torch
import torch.nn as nn
import torch.nn.functional as F

def calculateMLPAttn(h_i, h_j, W, a, negative_slope=0.2):
    """
    Computes alpha_ij based on the formula:
    alpha_ij = exp(LeakyReLU(a[Wh_i || Wh_j])) / sum(exp(LeakyReLU(a[Wh_i || Wh_k])))

    Inputs:
        h_i: Features of target node i [batch, d_in]
        h_j: Features of neighbor node j [batch, d_in]
        W:   Weight matrix for message projection [d_in, d_msg]
        a:   Weight vector of the attention MLP [1, 2 * d_msg]
    """

    # 1. Linear Projection: W * h
    Wh_i=torch.matmul(h_i,W)
    Wh_j=torch.matmul(h_j,W)

    # 2. Concatenation: [Wh_i || Wh_j]
    combined_features=torch.cat([Wh_i, Wh_j], dim=-1)

    # 3. MLP Weight application: a * [Wh_i || Wh_j]
    e_ij = torch.matmul(combined_features, a.t())

    # 4. LeakyReLU Activation
    e_ij = F.leaky_relu(e_ij, negative_slope=negative_slope)
    return e_ij

In [18]:
# Features for Node i and Node j
h_i = torch.tensor([1.0, 0.0, 2.0])
h_j = torch.tensor([0.0, 1.0, 0.5])

# Weight Matrix W (3x2)
W = torch.tensor([[0.1, 0.2],
                  [0.3, 0.4],
                  [0.5, 0.6]])

# Attention Weight Vector a (1x4)
a = torch.tensor([[0.2, 0.1, 0.3, 0.4]])
alpha_ij=calculateMLPAttn(h_i, h_j, W, a)
print(alpha_ij)

tensor([0.8050])



Once we obtain all attention factors, we can calculate the output features for each node by performing the weighted average:

$$h'_i = \sigma \left( \sum_{j \in \mathcal{N}_i} \alpha_{ij} \mathbf{W}h_j \right)$$

![image.png](https://uvadlc-notebooks.readthedocs.io/en/latest/_images/graph_attention.jpeg)

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def newFeaturesOfNode(v_j, alpha_ij, sigma=F.softmax):
    """
    Computes the output features h'_i for a node based on its neighbors.
    Formula: h'_i = sigma( sum_{j in Ni} alpha_ij * W * h_j )

    Inputs:
        v_j:      Projected features (W * h_j) of all neighbors.
                  Shape: [num_neighbors, d_msg]
        alpha_ij: Normalized attention weights for each neighbor.
                  Shape: [num_neighbors, 1]
        sigma:    Activation function (typically ELU in GAT).

    Returns:
        h_prime_i: The updated feature vector for node i.
    """

    # 1. Weighted Average: sum(alpha_ij * (W * h_j))
    weighted_features = alpha_ij * v_j

    # 2. Summation over the neighborhood (j in Ni)
    sum_aggregate = torch.sum(weighted_features, dim=0)

    # 3. Apply Softmax (sigma)
    h_prime_i = sigma(sum_aggregate)

    return h_prime_i

In [20]:
Wh_i=torch.matmul(h_i,W).unsqueeze(0)
Wh_j=torch.matmul(h_j,W).unsqueeze(0)

alpha_ii=calculateMLPAttn(h_i, h_i, W, a)
alpha_ij=calculateMLPAttn(h_j, h_i, W, a)

alphas= F.softmax(torch.cat([alpha_ii, alpha_ij], dim=0),dim=0)

W_neighbour=torch.cat([Wh_i,Wh_j],dim=0)

h_prime_0 = newFeaturesOfNode(W_neighbour, alphas)

print(f"New Features for Node 0: {h_prime_0}")


New Features for Node 0: tensor([0.4858, 0.5142])


/tmp/ipykernel_5951/584051724.py:28: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  h_prime_i = sigma(sum_aggregate)


## After having discussed the graph attention layer in detail, we can implement it below:

In [21]:
class GATLayer(nn.Module):

    def __init__(self, c_in, c_out, num_heads=1, concat_heads=True, alpha=0.2):
        """
        Inputs:
            c_in - Dimensionality of input features
            c_out - Dimensionality of output features
            num_heads - Number of heads, i.e. attention mechanisms to apply in parallel. The
                        output features are equally split up over the heads if concat_heads=True.
            concat_heads - If True, the output of the different heads is concatenated instead of averaged.
            alpha - Negative slope of the LeakyReLU activation.
        """
        super().__init__()
        self.num_heads = num_heads
        self.concat_heads = concat_heads
        if self.concat_heads:
            assert c_out % num_heads == 0, "Number of output features must be a multiple of the count of heads."
            c_out = c_out // num_heads

        # Sub-modules and parameters needed in the layer
        self.projection = nn.Linear(c_in, c_out * num_heads)
        self.a = nn.Parameter(torch.Tensor(num_heads, 2 * c_out)) # One per head
        self.leakyrelu = nn.LeakyReLU(alpha)

        # Initialization from the original implementation
        nn.init.xavier_uniform_(self.projection.weight.data, gain=1.414)
        nn.init.xavier_uniform_(self.a.data, gain=1.414)

    def forward(self, node_feats, adj_matrix, print_attn_probs=False):
        """
        Inputs:
            node_feats - Input features of the node. Shape: [batch_size, c_in]
            adj_matrix - Adjacency matrix including self-connections. Shape: [batch_size, num_nodes, num_nodes]
            print_attn_probs - If True, the attention weights are printed during the forward pass (for debugging purposes)
        """
        batch_size, num_nodes = node_feats.size(0), node_feats.size(1)

        # Apply linear layer and sort nodes by head
        node_feats = self.projection(node_feats)
        node_feats = node_feats.view(batch_size, num_nodes, self.num_heads, -1)

        # We need to calculate the attention logits for every edge in the adjacency matrix
        # Doing this on all possible combinations of nodes is very expensive
        # => Create a tensor of [W*h_i||W*h_j] with i and j being the indices of all edges
        edges = adj_matrix.nonzero(as_tuple=False) # Returns indices where the adjacency matrix is not 0 => edges
        node_feats_flat = node_feats.view(batch_size * num_nodes, self.num_heads, -1)
        edge_indices_row = edges[:,0] * num_nodes + edges[:,1]
        edge_indices_col = edges[:,0] * num_nodes + edges[:,2]
        a_input = torch.cat([
            torch.index_select(input=node_feats_flat, index=edge_indices_row, dim=0),
            torch.index_select(input=node_feats_flat, index=edge_indices_col, dim=0)
        ], dim=-1) # Index select returns a tensor with node_feats_flat being indexed at the desired positions along dim=0

        # Calculate attention MLP output (independent for each head)
        attn_logits = torch.einsum('bhc,hc->bh', a_input, self.a)
        attn_logits = self.leakyrelu(attn_logits)

        # Map list of attention values back into a matrix
        attn_matrix = attn_logits.new_zeros(adj_matrix.shape+(self.num_heads,)).fill_(-9e15)
        attn_matrix[adj_matrix[...,None].repeat(1,1,1,self.num_heads) == 1] = attn_logits.reshape(-1)

        # Weighted average of attention
        attn_probs = F.softmax(attn_matrix, dim=2)
        if print_attn_probs:
            print("Attention probs\n", attn_probs.permute(0, 3, 1, 2))
        node_feats = torch.einsum('bijh,bjhc->bihc', attn_probs, node_feats)

        # If heads should be concatenated, we can do this by reshaping. Otherwise, take mean
        if self.concat_heads:
            node_feats = node_feats.reshape(batch_size, num_nodes, -1)
        else:
            node_feats = node_feats.mean(dim=2)

        return node_feats

![image.png](https://zoshs2.github.io/assets/img/post/gcn_implement/GCN_message_passing.png)
we can apply the graph attention layer on our example graph above to understand the dynamics better.

In [22]:
layer = GATLayer(2, 2, num_heads=2)
layer.projection.weight.data = torch.Tensor([[1., 0.], [0., 1.]])
layer.projection.bias.data = torch.Tensor([0., 0.])
layer.a.data = torch.Tensor([[-0.2, 0.3], [0.1, -0.1]])

with torch.no_grad():
    out_feats = layer(node_feats, adj_matrix, print_attn_probs=True)

print("Adjacency matrix", adj_matrix)
print("Input features", node_feats)
print("Output features", out_feats)

Attention probs
 tensor([[[[0.3543, 0.6457, 0.0000, 0.0000],
          [0.1096, 0.1450, 0.2642, 0.4813],
          [0.0000, 0.1858, 0.2885, 0.5257],
          [0.0000, 0.2391, 0.2696, 0.4913]],

         [[0.5100, 0.4900, 0.0000, 0.0000],
          [0.2975, 0.2436, 0.2340, 0.2249],
          [0.0000, 0.3838, 0.3142, 0.3019],
          [0.0000, 0.4018, 0.3289, 0.2693]]]])
Adjacency matrix tensor([[[1., 1., 0., 0.],
         [1., 1., 1., 1.],
         [0., 1., 1., 1.],
         [0., 1., 1., 1.]]])
Input features tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
Output features tensor([[[1.2913, 1.9800],
         [4.2344, 3.7725],
         [4.6798, 4.8362],
         [4.5043, 4.7351]]])


As before, the input layer is initialized as an identity matrix, but we set `a`
 to be a vector of arbitrary numbers to obtain different attention values. We use two heads to show the parallel, independent attention mechanisms working in the layer.